# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Davionnic/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Ranked actions + reason codes

**Lane (locked):** Refresh / Content Opportunity Scoring.

**Product artifact:** a ranked review queue — each row gets a **suggested action**, **reason codes**, and a **confidence** band. The teaching label is still `is_declining_label = (trend_direction == "down")`. Ranking quality is measured on **client-holdout Precision@50** (same as ML-07 / ML-08 / ML-09).

**Action vocabulary (decision-support, not auto-publish):**

| Action | When it fires (reason-driven) | What a reviewer should do |
|---|---|---|
| `refresh_and_review_ctr` | decline / model risk **and** CTR review candidate | Open page; check title/meta/snippet vs intent before rewriting body |
| `refresh_and_review_engagement` | decline / model risk **and** low engagement | Check on-page experience (scroll / engagement), not only copy length |
| `expand_and_refresh` | thin visible page | Expand thin content **then** refresh; do not ship a shallow rewrite |
| `refresh` | decline / stale / visible model opportunity without CTR/engagement flags | Standard content refresh review |
| `monitor` | no strong refresh reasons | Leave alone; watch next window |

**Measured ranking (this notebook run, client-holdout, `RANDOM_STATE=42`, same split as ML-07/08/09):** Random Forest Precision@50 vs transparent baseline — printed live below (lane receipt ≈ **0.680** vs **0.240**). Claims stay **decision-support ranking**, not causal SEO.


In [1]:
import os
import json
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
import sklearn

# walk up until we find the repo root (works from work/notebooks/ or Colab)
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

ROOT = Path(".").resolve()
sys.path.insert(0, str(ROOT / "scripts"))
from ml_utils import (  # noqa: E402
    MODEL_CATEGORICAL_FEATURES,
    MODEL_NUMERIC_FEATURES,
    normalize,
    percentile_rank,
    precision_at_k,
)

OUT = ROOT / "work" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

RAW = "data/raw/content_refresh_anonymized.csv"
FEAT = "data/processed/refresh_feature_vector.csv"
assert os.path.exists(RAW), f"missing starter CSV at {os.path.abspath(RAW)}"

if not os.path.exists(FEAT):
    import runpy

    runpy.run_path("scripts/01_prepare_features.py", run_name="__main__")

df = pd.read_csv(FEAT)
assert "is_declining_label" in df.columns
RANDOM_STATE = 42

print("Working dir:", os.getcwd())
print(f"feature rows={len(df):,}  cols={df.shape[1]}  clients={df['client_id'].nunique()}")
print("lane: Refresh / Content Opportunity Scoring")
print("label: is_declining_label = (trend_direction == 'down')")
print("metric: Precision@50 on client-holdout (RANDOM_STATE=42)")
print(f"sklearn={sklearn.__version__}  numpy={np.__version__}  pandas={pd.__version__}")
print("artifact: ranked action playbook (actions + reason codes + confidence)")


def build_feature_matrix(frame: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    numeric_features = [c for c in MODEL_NUMERIC_FEATURES if c in frame.columns]
    categorical_features = [c for c in MODEL_CATEGORICAL_FEATURES if c in frame.columns]

    numeric_frame = frame[numeric_features].apply(pd.to_numeric, errors="coerce")
    numeric_frame = numeric_frame.replace([np.inf, -np.inf], np.nan).fillna(0)

    categorical_frame = frame[categorical_features].fillna("unknown").astype(str)
    encoded_frame = pd.get_dummies(
        categorical_frame,
        prefix=categorical_features,
        dummy_na=False,
        dtype=float,
    )

    feature_frame = pd.concat(
        [numeric_frame.reset_index(drop=True), encoded_frame.reset_index(drop=True)],
        axis=1,
    )
    forbidden = {"trend_direction", "trend_pct", "is_declining_label"}
    overlap = forbidden.intersection(feature_frame.columns)
    assert not overlap, f"leakage columns in X: {overlap}"
    return feature_frame, list(feature_frame.columns)


def make_client_holdout(frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, set[str]]:
    # Identical to ML-07/08 / scripts/03_train_model.py (default_rng + first-seen client order).
    all_indices = np.arange(len(frame))
    client_series = frame["client_id"].fillna("unknown").astype(str)
    unique_clients = client_series.drop_duplicates().to_numpy()
    rng = np.random.default_rng(RANDOM_STATE)
    shuffled = rng.permutation(unique_clients)
    test_client_count = max(1, int(round(len(shuffled) * 0.2)))
    test_clients = set(shuffled[:test_client_count].tolist())
    test_mask = client_series.isin(test_clients).to_numpy()
    return all_indices[~test_mask], all_indices[test_mask], test_clients


def baseline_refresh_score(frame: pd.DataFrame) -> pd.Series:
    visibility = percentile_rank(np.log1p(frame["impressions_90d"]))
    freshness = percentile_rank(frame["days_since_last_update"])
    position_opportunity = (
        (1 - normalize(frame["avg_position"].clip(lower=1, upper=50)))
        * visibility
        * (frame["avg_position"] > 0).astype(int)
    )
    depth_gap = (1 - percentile_rank(frame["word_count"])) * visibility
    return (
        0.40 * visibility
        + 0.30 * freshness
        + 0.25 * position_opportunity
        + 0.05 * depth_gap
    ).clip(0, 1)


def baseline_reason_codes(row: pd.Series) -> list[str]:
    reasons: list[str] = []
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        reasons.append("stale_visible_page")
    if str(row["trend_direction"]).lower() == "down" and row["impressions_90d"] >= 100:
        reasons.append("declining_with_demand")
    if row["word_count"] > 0 and row["word_count"] < 1200 and row["impressions_90d"] >= 250:
        reasons.append("thin_visible_page")
    if row["avg_position"] > 0 and row["avg_position"] <= 10 and row["content_age_days"] >= 180:
        reasons.append("page_one_decay_risk")
    if row["impressions_90d"] >= 500 and 0 < row["avg_position"] <= 20 and row["ctr"] < 0.5:
        reasons.append("low_ctr_visible_page")
    if row["sessions_90d"] >= 30 and (
        (row["engagement_rate"] > 0 and row["engagement_rate"] < 30)
        or (row["scroll_rate"] > 0 and row["scroll_rate"] < 30)
    ):
        reasons.append("low_engagement_visible_page")
    if not reasons:
        reasons.append("general_refresh_review")
    return reasons


def merged_reason_codes(row: pd.Series) -> str:
    reasons = list(row["_base_reasons"])
    if row["best_model_probability"] >= 0.65:
        reasons.append("model_decline_risk")
    if row["best_model_probability"] >= 0.5 and row["impressions_90d"] >= 500:
        reasons.append("visible_model_opportunity")
    if (
        row["impressions_90d"] >= 500
        and row["avg_position"] > 0
        and row["avg_position"] <= 20
        and row["ctr"] < 0.5
    ):
        reasons.append("ctr_review_candidate")
    if row["sessions_90d"] >= 30 and (
        (row["engagement_rate"] > 0 and row["engagement_rate"] < 30)
        or (row["scroll_rate"] > 0 and row["scroll_rate"] < 30)
    ):
        reasons.append("engagement_review_candidate")
    unique = []
    for r in reasons:
        if r not in unique:
            unique.append(r)
    return "|".join(unique or ["general_refresh_review"])


def suggested_action(row: pd.Series) -> str:
    reasons = set(str(row["final_reason_codes"]).split("|"))
    if "thin_visible_page" in reasons:
        return "expand_and_refresh"
    if "ctr_review_candidate" in reasons and (
        "model_decline_risk" in reasons or "declining_with_demand" in reasons
    ):
        return "refresh_and_review_ctr"
    if "engagement_review_candidate" in reasons and (
        "model_decline_risk" in reasons or "declining_with_demand" in reasons
    ):
        return "refresh_and_review_engagement"
    if {
        "model_decline_risk",
        "declining_with_demand",
        "stale_visible_page",
        "visible_model_opportunity",
    }.intersection(reasons):
        return "refresh"
    return "monitor"


def confidence_label(row: pd.Series, high_threshold: float, medium_threshold: float) -> str:
    if (
        row["final_refresh_score"] >= high_threshold
        and row["impressions_90d"] >= 500
        and row["sessions_90d"] >= 10
        and row["best_model_probability"] >= 0.5
    ):
        return "high"
    if row["final_refresh_score"] >= medium_threshold:
        return "medium"
    return "low"


# --- recompute holdout Prec@50 (never invent) ---
X_all, feature_columns = build_feature_matrix(df)
y_all = df["is_declining_label"].astype(int)
train_idx, test_idx, test_clients = make_client_holdout(df)
test_clients = sorted(str(c) for c in test_clients)

rf = RandomForestClassifier(
    class_weight="balanced_subsample",
    max_depth=10,
    min_samples_leaf=25,
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf.fit(X_all.iloc[train_idx], y_all.iloc[train_idx])
proba_holdout = rf.predict_proba(X_all.iloc[test_idx])[:, 1]
y_holdout = y_all.iloc[test_idx].to_numpy()

base_full = baseline_refresh_score(df)
base_holdout = base_full.iloc[test_idx].to_numpy()

rf_p50 = float(precision_at_k(y_holdout, proba_holdout, 50))
rf_p20 = float(precision_at_k(y_holdout, proba_holdout, 20))
base_p50 = float(precision_at_k(y_holdout, base_holdout, 50))
base_p20 = float(precision_at_k(y_holdout, base_holdout, 20))
rf_ap = float(average_precision_score(y_holdout, proba_holdout))
rf_auc = float(roc_auc_score(y_holdout, proba_holdout))
base_rate_holdout = float(y_holdout.mean())
top50_pos = int(pd.Series(y_holdout).iloc[np.argsort(-proba_holdout)[:50]].sum())

print("\n=== ranking quality (client-holdout, live) ===")
print(f"test_clients={test_clients}")
print(f"train_rows={len(train_idx):,}  test_rows={len(test_idx):,}")
print(f"holdout base rate = {base_rate_holdout:.3f}")
print(f"baseline Precision@50 = {base_p50:.3f}  (Precision@20 = {base_p20:.3f})")
print(f"random_forest Precision@50 = {rf_p50:.3f}  (Precision@20 = {rf_p20:.3f})")
print(f"random_forest AP = {rf_ap:.3f}  ROC-AUC = {rf_auc:.3f}")
print(f"holdout top-50 proxy positives = {top50_pos}/50")

# Score full inventory with the holdout-trained model (same pattern as scripts/03).
proba_all = rf.predict_proba(X_all)[:, 1]

queue = df.copy()
queue["baseline_refresh_score"] = base_full.values
queue["best_model_name"] = "random_forest"
queue["best_model_probability"] = proba_all
queue["_base_reasons"] = queue.apply(baseline_reason_codes, axis=1)
queue["final_reason_codes"] = queue.apply(merged_reason_codes, axis=1)
queue["suggested_action"] = queue.apply(suggested_action, axis=1)
queue["baseline_score_normalized"] = normalize(queue["baseline_refresh_score"])
queue["final_refresh_score"] = (
    100
    * (
        0.70 * queue["best_model_probability"]
        + 0.30 * queue["baseline_score_normalized"]
    )
).clip(0, 100)
high_thr = float(queue["final_refresh_score"].quantile(0.8))
med_thr = float(queue["final_refresh_score"].quantile(0.5))
queue["confidence"] = queue.apply(lambda r: confidence_label(r, high_thr, med_thr), axis=1)
queue = queue.sort_values(
    ["final_refresh_score", "impressions_90d", "sessions_90d"],
    ascending=[False, False, False],
).reset_index(drop=True)
queue["final_rank"] = queue.index + 1

action_counts = queue["suggested_action"].value_counts()
conf_counts = queue["confidence"].value_counts()
reason_counter = Counter()
for text in queue["final_reason_codes"]:
    for reason in str(text).split("|"):
        if reason:
            reason_counter[reason] += 1

print("\n=== playbook action mix (full inventory) ===")
print(action_counts.to_string())
print("\n=== confidence mix ===")
print(conf_counts.to_string())
print("\n=== top reason codes ===")
for reason, count in reason_counter.most_common(12):
    print(f"  {reason}: {count:,}")

preview_cols = [
    "final_rank",
    "final_refresh_score",
    "best_model_probability",
    "suggested_action",
    "confidence",
    "final_reason_codes",
    "impressions_90d",
    "sessions_90d",
    "is_declining_label",
]
print("\n=== top-15 queue preview (ids omitted; no URLs / client names) ===")
prev = queue[preview_cols].head(15).copy()
prev["final_reason_codes"] = prev["final_reason_codes"].str.replace("|", ", ", regex=False)
# drop client/content ids already omitted
print(prev.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

top20_actions = queue.head(20)["suggested_action"].value_counts().to_dict()
print("\ntop-20 suggested_action mix:", top20_actions)
print(f"top-50 full-queue proxy-positive rate = {queue.head(50)['is_declining_label'].mean():.3f}")


Working dir: /workspace/flyrank-ml-internship
feature rows=30,000  cols=52  clients=32
lane: Refresh / Content Opportunity Scoring
label: is_declining_label = (trend_direction == 'down')
metric: Precision@50 on client-holdout (RANDOM_STATE=42)
sklearn=1.9.1  numpy=2.5.3  pandas=3.0.6
artifact: ranked action playbook (actions + reason codes + confidence)



=== ranking quality (client-holdout, live) ===
test_clients=['client_0b918943df', 'client_1a6562590e', 'client_4fc82b26ae', 'client_98a3ab7c34', 'client_d4735e3a26', 'client_f74efabef1']
train_rows=27,675  test_rows=2,325
holdout base rate = 0.391
baseline Precision@50 = 0.240  (Precision@20 = 0.150)
random_forest Precision@50 = 0.680  (Precision@20 = 0.700)
random_forest AP = 0.610  ROC-AUC = 0.747
holdout top-50 proxy positives = 34/50



=== playbook action mix (full inventory) ===
suggested_action
monitor                          13264
refresh                           8040
refresh_and_review_ctr            6646
refresh_and_review_engagement     1968
expand_and_refresh                  82

=== confidence mix ===
confidence
low       15000
medium    11368
high       3632

=== top reason codes ===
  declining_with_demand: 13,152
  visible_model_opportunity: 10,590
  low_ctr_visible_page: 9,759
  ctr_review_candidate: 9,759
  general_refresh_review: 9,117
  model_decline_risk: 7,927
  page_one_decay_risk: 7,076
  low_engagement_visible_page: 6,508
  engagement_review_candidate: 6,508
  thin_visible_page: 82
  stale_visible_page: 17

=== top-15 queue preview (ids omitted; no URLs / client names) ===
 final_rank  final_refresh_score  best_model_probability       suggested_action confidence                                                                                                                                       

## 2. Intended use and limits

**Who uses this:** an SEO / content editor (or FlyRank reviewer) who needs a **first-pass review order** across a large inventory.

**What it is for:**
- Surface **which pages to open first** for refresh, CTR/snippet review, engagement review, expansion, or monitoring.
- Attach **readable reason codes** so a human can accept, reject, or re-route the suggestion.
- Prefer **high-confidence** rows when capacity is limited (visible + model-supported).

**Where it stops being valid:**
- **Not** a publish button — scores do not authorize automatic rewrites or CMS writes.
- **Not** causal SEO — Precision@50 says the ranker finds same-window proxy-declining pages; it does **not** prove a refresh will recover traffic.
- **Not** cross-portfolio gospel — client-holdout helps, but the starter CSV is one teaching slice (same-window label, no future outcome window).
- **Not** safe on raw titles/URLs/queries — playbook exports stay anonymized (`content_id` / aggregates only).
- **Not** a substitute for brand, legal, or YMYL editorial judgment.

**Claim stance:** observed / measured / directional / **decision-support ranking**.


In [2]:
# Intended-use card — numbers from this run only
use_card = {
    "lane": "Refresh / Content Opportunity Scoring",
    "primary_users": ["SEO editor", "content reviewer", "FlyRank analyst"],
    "intended_use": "Rank pages for human refresh review with action + reason codes",
    "not_for": [
        "automatic publishing / CMS writes",
        "causal claims that refresh recovers traffic",
        "scoring with private titles/URLs/queries in exports",
    ],
    "label": "is_declining_label = (trend_direction == 'down')",
    "split": "client_holdout",
    "random_state": RANDOM_STATE,
    "best_model": "random_forest",
    "precision_at_50_holdout": round(rf_p50, 3),
    "baseline_precision_at_50_holdout": round(base_p50, 3),
    "holdout_base_rate": round(base_rate_holdout, 3),
    "claim_stance": "decision-support ranking, not causal SEO",
}

print("=== intended use card ===")
for k, v in use_card.items():
    print(f"{k}: {v}")

# Capacity vignette: if a reviewer can only open 50 pages, what do they get?
cap = queue.head(50)
print("\n=== if capacity = top 50 only ===")
print("action mix:")
print(cap["suggested_action"].value_counts().to_string())
print("confidence mix:")
print(cap["confidence"].value_counts().to_string())
print(f"proxy-positive rate in top 50 (full queue order) = {cap['is_declining_label'].mean():.3f}")
print("Note: holdout Precision@50 is the honest generalization metric;")
print("full-queue top-50 positive rate is descriptive inventory triage, not the validation number.")


=== intended use card ===
lane: Refresh / Content Opportunity Scoring
primary_users: ['SEO editor', 'content reviewer', 'FlyRank analyst']
intended_use: Rank pages for human refresh review with action + reason codes
not_for: ['automatic publishing / CMS writes', 'causal claims that refresh recovers traffic', 'scoring with private titles/URLs/queries in exports']
label: is_declining_label = (trend_direction == 'down')
split: client_holdout
random_state: 42
best_model: random_forest
precision_at_50_holdout: 0.68
baseline_precision_at_50_holdout: 0.24
holdout_base_rate: 0.391
claim_stance: decision-support ranking, not causal SEO

=== if capacity = top 50 only ===
action mix:
suggested_action
refresh_and_review_ctr           30
refresh                          14
refresh_and_review_engagement     6
confidence mix:
confidence
high      38
medium    12
proxy-positive rate in top 50 (full queue order) = 1.000
Note: holdout Precision@50 is the honest generalization metric;
full-queue top-50 p

## 3. Human review + the no-go list

**Must check before acting (every high/medium suggestion):**
1. **Page still exists / still owned** — soft-404s and migrations break the row.
2. **Intent match** — does the URL still match the query cluster the metrics reflect?
3. **Reason-code sanity** — open the live SERP/snippet if `ctr_review_candidate`; scroll the page if `engagement_review_candidate`; measure depth if `thin_visible_page`.
4. **Seasonality / campaign context** — a "decline" during an expected off-season is not an urgent rewrite.
5. **Duplicate / cannibalization** — two high-score siblings may need merge/prune, not two refreshes.
6. **Brand / legal / YMYL** — humans own the final copy decision.

**No-go list (never automate):**
- Auto-publishing model-written rewrites from the score alone.
- Deleting or deindexing from `monitor` / low-confidence rows.
- Using `trend_direction` / `trend_pct` / label siblings as model **features** (leakage).
- Shipping client names, raw URLs, or private queries in public notebooks / paper exports.
- Claiming "this refresh will recover traffic" from Precision@K alone.
- Blind bulk actions on an entire client without sampling review.

The model **ranks and explains**; the human **decides**.


In [3]:
# Operational checklist backed by this queue's high-confidence slice
high = queue[queue["confidence"] == "high"].copy()
print(f"high-confidence rows = {len(high):,}")
print("high-confidence action mix:")
print(high["suggested_action"].value_counts().to_string())

# How often would a CTR-flagged action also carry engagement review?
both = high["final_reason_codes"].str.contains("ctr_review_candidate") & high[
    "final_reason_codes"
].str.contains("engagement_review_candidate")
print(f"\nhigh-conf rows with BOTH ctr + engagement review reasons = {int(both.sum()):,}")

# Sample human-review packet (anonymized): top 5 high-confidence only
packet_cols = [
    "final_rank",
    "suggested_action",
    "confidence",
    "final_refresh_score",
    "best_model_probability",
    "final_reason_codes",
    "impressions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "word_count",
]
packet = high[packet_cols].head(5).copy()
packet["final_reason_codes"] = packet["final_reason_codes"].str.replace("|", ", ", regex=False)
print("\n=== sample human-review packet (top 5 high-confidence, anonymized) ===")
print(packet.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

no_go = [
    "auto_publish_from_score",
    "bulk_delete_from_monitor",
    "train_with_trend_pct_or_label_siblings",
    "export_client_names_urls_queries",
    "causal_traffic_recovery_claim_from_precision_at_k",
]
print("\nno-go automations:", no_go)


high-confidence rows = 3,632
high-confidence action mix:
suggested_action
refresh_and_review_ctr           1878
refresh_and_review_engagement     941
refresh                           805
expand_and_refresh                  8

high-conf rows with BOTH ctr + engagement review reasons = 793

=== sample human-review packet (top 5 high-confidence, anonymized) ===
 final_rank       suggested_action confidence  final_refresh_score  best_model_probability                                                                                                                                                         final_reason_codes  impressions_90d  avg_position   ctr  content_age_days  days_since_last_update  word_count
          1 refresh_and_review_ctr       high               81.217                   0.785                                                           declining_with_demand, low_ctr_visible_page, model_decline_risk, visible_model_opportunity, ctr_review_candidate             8064       

## 4. Monitoring / retrain triggers

Recommendations go **stale** when the world or the proxy drifts. Watch these triggers; any one is enough to pause bulk use and re-check.

| Trigger | Why it matters | Suggested response |
|---|---|---|
| Holdout / fresh-client **Precision@50** falls toward baseline (~0.24) or base rate | Ranker lost lift | Retrain + re-run leakage checklist; do not raise automation |
| Action mix shifts sharply (e.g. `monitor` collapses or `expand_and_refresh` explodes) | Thresholds or inventory composition changed | Recalibrate reason thresholds; sample-review 20 rows |
| Label definition changes (new trend window, future-outcome label) | Old model answers a different question | New data contract + full retrain |
| Feature pipeline change (new columns, fill rules) | Silent schema drift | Freeze queue; rebuild feature vector; compare Prec@K |
| Client mix changes a lot (new verticals) | Client-holdout assumptions weaken | Re-split, re-evaluate by client cohort |
| Editors reject >~30% of high-confidence suggestions in spot checks | Score ≠ editorial truth | Raise human-review bar; inspect false-positive patterns |
| Seasonality / algorithm-update weeks | Same-window proxy misreads shocks | Temporary monitor-only mode |

**Cadence (practical):** weekly spot-check of top 20; monthly Prec@K on a fresh client-holdout; retrain when Prec@50 lift vs baseline erodes or schema/label changes.


In [4]:
# Monitoring dashboard snapshot from this run (receipt for the paper)
mon = {
    "precision_at_50_rf_holdout": round(rf_p50, 3),
    "precision_at_50_baseline_holdout": round(base_p50, 3),
    "lift_vs_baseline_p50": round(rf_p50 / base_p50, 3) if base_p50 else None,
    "holdout_base_rate": round(base_rate_holdout, 3),
    "action_mix": {k: int(v) for k, v in action_counts.items()},
    "confidence_mix": {k: int(v) for k, v in conf_counts.items()},
    "high_conf_share": round(float((queue["confidence"] == "high").mean()), 3),
    "monitor_share": round(float((queue["suggested_action"] == "monitor").mean()), 3),
    "retrain_if_p50_near_baseline": round(base_p50, 3),
    "retrain_if_p50_near_base_rate": round(base_rate_holdout, 3),
}

print("=== monitoring snapshot ===")
for k, v in mon.items():
    print(f"{k}: {v}")

# Simple drift alarms against this run's own baselines
alarms = []
if rf_p50 <= base_p50 + 0.05:
    alarms.append("ALERT: model Prec@50 within 0.05 of baseline — investigate before acting")
if rf_p50 < base_rate_holdout:
    alarms.append("ALERT: model Prec@50 below holdout base rate — do not use queue")
if (queue["suggested_action"] == "monitor").mean() < 0.2:
    alarms.append("ALERT: monitor share collapsed — reason thresholds may be over-firing")
if not alarms:
    alarms.append("no alarm on this teaching run (Prec@50 lift intact; action mix sane)")
print("\nalarms:")
for a in alarms:
    print("-", a)


=== monitoring snapshot ===
precision_at_50_rf_holdout: 0.68
precision_at_50_baseline_holdout: 0.24
lift_vs_baseline_p50: 2.833
holdout_base_rate: 0.391
action_mix: {'monitor': 13264, 'refresh': 8040, 'refresh_and_review_ctr': 6646, 'refresh_and_review_engagement': 1968, 'expand_and_refresh': 82}
confidence_mix: {'low': 15000, 'medium': 11368, 'high': 3632}
high_conf_share: 0.121
monitor_share: 0.442
retrain_if_p50_near_baseline: 0.24
retrain_if_p50_near_base_rate: 0.391

alarms:
- no alarm on this teaching run (Prec@50 lift intact; action mix sane)


## 5. Exports for the paper

Write the playbook receipts under `work/outputs/` (JSON + markdown; CSVs stay gitignored). Figures the paper can reuse: action mix and top reason codes.


In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# --- metrics JSON (committed receipt) ---
playbook_metrics = {
    "assignment": "ML-10",
    "lane": "Refresh / Content Opportunity Scoring",
    "label": "is_declining_label = (trend_direction == 'down')",
    "split": "client_holdout",
    "random_state": RANDOM_STATE,
    "test_clients": test_clients,
    "train_rows": int(len(train_idx)),
    "test_rows": int(len(test_idx)),
    "best_model": "random_forest",
    "precision_at_20_holdout": round(rf_p20, 3),
    "precision_at_50_holdout": round(rf_p50, 3),
    "average_precision_holdout": round(rf_ap, 3),
    "roc_auc_holdout": round(rf_auc, 3),
    "baseline_precision_at_20_holdout": round(base_p20, 3),
    "baseline_precision_at_50_holdout": round(base_p50, 3),
    "holdout_base_rate": round(base_rate_holdout, 3),
    "holdout_top50_proxy_positives": top50_pos,
    "rows_scored": int(len(queue)),
    "action_mix": {k: int(v) for k, v in action_counts.items()},
    "confidence_mix": {k: int(v) for k, v in conf_counts.items()},
    "top_reason_codes": [
        {"reason": r, "count": int(c)} for r, c in reason_counter.most_common(12)
    ],
    "score_blend": {"model_probability": 0.70, "baseline_normalized": 0.30},
    "claim_stance": "decision-support ranking, not causal SEO",
    "sklearn_version": sklearn.__version__,
}

metrics_path = OUT / "action_playbook_metrics.json"
metrics_path.write_text(json.dumps(playbook_metrics, indent=2) + "\n")
print("wrote", metrics_path)

# --- markdown playbook ---
lines = [
    "# Content Action Playbook — Refresh / Content Opportunity Scoring",
    "",
    "Decision-support ranking for which page a reviewer should open first.",
    "Not causal SEO; not an auto-publish system.",
    "",
    "## Ranking quality (client-holdout)",
    "",
    f"- Best model: `random_forest`",
    f"- Precision@50: **{rf_p50:.3f}** (baseline **{base_p50:.3f}**, holdout base rate **{base_rate_holdout:.3f}**)",
    f"- Precision@20: **{rf_p20:.3f}** (baseline **{base_p20:.3f}**)",
    f"- Holdout top-50 proxy positives: **{top50_pos}/50**",
    "",
    "## Suggested actions",
    "",
]
for action, count in action_counts.items():
    lines.append(f"- `{action}`: {int(count):,}")
lines += [
    "",
    "## Confidence mix",
    "",
]
for band, count in conf_counts.items():
    lines.append(f"- `{band}`: {int(count):,}")
lines += [
    "",
    "## Top reason codes",
    "",
]
for reason, count in reason_counter.most_common(10):
    lines.append(f"- `{reason}`: {int(count):,}")
lines += [
    "",
    "## Human review (short)",
    "",
    "Verify live page, intent, seasonality, duplicates, and brand/legal before acting.",
    "Never auto-publish from the score alone.",
    "",
    "## Monitoring triggers",
    "",
    "- Prec@50 approaches baseline or base rate → retrain / pause automation.",
    "- Action-mix shock or high editor-reject rate → recalibrate + sample review.",
    "- Label or feature-schema change → new contract + full rebuild.",
    "",
    "## Claim stance",
    "",
    "observed / measured / directional / decision-support ranking.",
    "",
]
md_path = OUT / "action_playbook.md"
md_path.write_text("\n".join(lines))
print("wrote", md_path)

# --- charts ---
fig_dir = OUT / "charts"
fig_dir.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 4))
actions_ordered = action_counts.sort_values(ascending=True)
ax.barh(actions_ordered.index.astype(str), actions_ordered.values, color="#426B69")
ax.set_title("Suggested action mix (playbook)")
ax.set_xlabel("pages")
fig.tight_layout()
action_fig = fig_dir / "playbook_action_mix.png"
fig.savefig(action_fig, dpi=120)
plt.close(fig)
print("wrote", action_fig)

fig, ax = plt.subplots(figsize=(8, 4))
top_r = reason_counter.most_common(8)
ax.barh([r for r, _ in top_r][::-1], [c for _, c in top_r][::-1], color="#6B4242")
ax.set_title("Top reason codes (playbook)")
ax.set_xlabel("pages")
fig.tight_layout()
reason_fig = fig_dir / "playbook_reason_codes.png"
fig.savefig(reason_fig, dpi=120)
plt.close(fig)
print("wrote", reason_fig)

# anonymized top-20 table for paper (markdown only — no CSV commit)
top_md = OUT / "action_playbook_top20.md"
t20 = queue.head(20)[
    [
        "final_rank",
        "final_refresh_score",
        "best_model_probability",
        "suggested_action",
        "confidence",
        "final_reason_codes",
        "impressions_90d",
        "is_declining_label",
    ]
].copy()
with top_md.open("w") as f:
    f.write("# Top 20 playbook rows (anonymized)\n\n")
    f.write("| Rank | Score | P(model) | Action | Confidence | Reasons | Impr.90d | Proxy↓ |\n")
    f.write("|---:|---:|---:|---|---|---|---:|---:|\n")
    for _, row in t20.iterrows():
        reasons = str(row.final_reason_codes).replace("|", ", ")
        f.write(
            f"| {int(row.final_rank)} | {row.final_refresh_score:.1f} | "
            f"{row.best_model_probability:.3f} | `{row.suggested_action}` | "
            f"{row.confidence} | {reasons} | {int(row.impressions_90d)} | "
            f"{int(row.is_declining_label)} |\n"
        )
print("wrote", top_md)
print("\nExports ready for the paper under work/outputs/ (capstone notebook untouched).")


wrote /workspace/flyrank-ml-internship/work/outputs/action_playbook_metrics.json


wrote /workspace/flyrank-ml-internship/work/outputs/action_playbook.md
wrote /workspace/flyrank-ml-internship/work/outputs/charts/playbook_action_mix.png
wrote /workspace/flyrank-ml-internship/work/outputs/charts/playbook_reason_codes.png
wrote /workspace/flyrank-ml-internship/work/outputs/action_playbook_top20.md

Exports ready for the paper under work/outputs/ (capstone notebook untouched).


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Receipt:** `work/outputs/action_playbook_metrics.json`, `work/outputs/action_playbook.md`, `work/outputs/action_playbook_top20.md`, and charts under `work/outputs/charts/`. Capstone notebook untouched.

*(Portal submit is out of scope for this agent run — commit + push only.)*
